# PathSnap: LightGBM and CatBoost on candidate rows
**Input files** (put them in the same folder as this notebook, or in `/content` on Colab):
- `train_candidates.csv` (or `.csv.gz`): trip_id, observation_index, arc_id, 18 features, label
- `student_test_candidates.csv` (or `.csv.gz`): same columns
- `student_test_references.csv`: trip_id, observation_index, reference_arc_id (blank = uncertain)

Training uses train only. Student_test is only used to report results.

In [ ]:
!pip -q install lightgbm catboost

In [ ]:
import os, time, numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss, confusion_matrix
import lightgbm as lgb
from catboost import CatBoostClassifier

def find(name):
    for p in (name, name + ".gz", "/content/" + name, "/content/" + name + ".gz"):
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
FEATURES = ['dist_m','dist_min_m','dist_rel_m','rank_dist','n_cands','proj_frac','arc_len_m','dist_end_m',
            'head_diff_deg','head_cos','speed_mps','moving','dt_prev_s','dist_prev_pt_m','dist_next_pt_m',
            'has_reverse','deg_from','deg_to']
tr = pd.read_csv(find("train_candidates.csv"), dtype={"trip_id": str})
te = pd.read_csv(find("student_test_candidates.csv"), dtype={"trip_id": str})
ref = pd.read_csv(find("student_test_references.csv"), dtype={"trip_id": str})
print(tr.shape, te.shape, ref.shape, "| positive rate (train):", round(tr.label.mean(), 4))

(1966581, 22) (233145, 22) (19160, 3) | positive rate (train): 0.0943


In [ ]:
# Evaluation helpers
ref_ok = ref.dropna(subset=["reference_arc_id"]).astype({"reference_arc_id": int})

def top1_and_miss(df, score):
    d = df[["trip_id", "observation_index", "arc_id"]].copy(); d["score"] = score
    best = d.sort_values("score", ascending=False).drop_duplicates(["trip_id", "observation_index"])
    m = ref_ok.merge(best, on=["trip_id", "observation_index"], how="left")            # certain references only
    top1 = (m.arc_id == m.reference_arc_id).mean()
    cand_set = d.merge(ref_ok, left_on=["trip_id", "observation_index", "arc_id"],
                       right_on=["trip_id", "observation_index", "reference_arc_id"], how="inner")
    miss = 1 - len(cand_set[["trip_id", "observation_index"]].drop_duplicates()) / len(ref_ok)
    return top1, miss

def evaluate(name, fit_fn, pred_fn):
    t0 = time.time(); model = fit_fn(tr[FEATURES], tr.label); t_fit = time.time() - t0
    t0 = time.time(); p = pred_fn(model, te[FEATURES]); t_inf = time.time() - t0
    y = te.label.values; yhat = (p >= 0.5).astype(int); top1, miss = top1_and_miss(te, p)
    r = dict(model=name, accuracy=accuracy_score(y, yhat), precision=precision_score(y, yhat, zero_division=0),
             recall=recall_score(y, yhat), f1=f1_score(y, yhat), log_loss=log_loss(y, p),
             top1_obs_acc=top1, candidate_miss_rate=miss, train_s=t_fit, infer_s=t_inf)
    print(name, "confusion matrix [[TN FP][FN TP]]:\n", confusion_matrix(y, yhat)); return r, model, p

In [ ]:
# 1. See where the NaNs are
print("NaN per feature (train):\n", tr[FEATURES].isna().sum()[lambda s: s > 0])
print("NaN per feature (test):\n", te[FEATURES].isna().sum()[lambda s: s > 0])

# 2. Fix: infinities -> NaN, then fill using TRAIN medians only (no leakage)
for d in (tr, te):
    d[FEATURES] = d[FEATURES].replace([np.inf, -np.inf], np.nan)
med = tr[FEATURES].median()
tr[FEATURES] = tr[FEATURES].fillna(med)
te[FEATURES] = te[FEATURES].fillna(med)

# 3. Drop rows with a missing label, then confirm
tr = tr.dropna(subset=["label"]).reset_index(drop=True)
te = te.dropna(subset=["label"]).reset_index(drop=True)
tr["label"] = tr["label"].astype(int); te["label"] = te["label"].astype(int)
assert not tr[FEATURES].isna().any().any() and not te[FEATURES].isna().any().any()
print("clean:", tr.shape, te.shape)

NaN per feature (train):
 Series([], dtype: int64)
NaN per feature (test):
 Series([], dtype: int64)
clean: (1966581, 22) (233145, 22)


In [ ]:
# LightGBM
POS_W = (tr.label == 0).sum() / (tr.label == 1).sum()      # class imbalance weight
lgb_fit = lambda X, y: lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63, subsample=0.8,
              colsample_bytree=0.8, scale_pos_weight=1.0, random_state=42, n_jobs=-1, verbose=-1).fit(X, y)
r_lgb, m_lgb, p_lgb = evaluate("LightGBM", lgb_fit, lambda m, X: m.predict_proba(X)[:, 1])

# CatBoost
cb_fit = lambda X, y: CatBoostClassifier(iterations=500, learning_rate=0.08, depth=8, random_seed=42,
              verbose=0).fit(X, y)
r_cb, m_cb, p_cb = evaluate("CatBoost", cb_fit, lambda m, X: m.predict_proba(X)[:, 1])
res = pd.DataFrame([r_lgb, r_cb]).set_index("model").round(4); res

LightGBM confusion matrix [[TN FP][FN TP]]:
 [[211948   2062]
 [  2763  16372]]
CatBoost confusion matrix [[TN FP][FN TP]]:
 [[212192   1818]
 [  2683  16452]]


,accuracy,precision,recall,f1,log_loss,top1_obs_acc,candidate_miss_rate,train_s,infer_s
model,,,,,,,,,
LightGBM,0.9793,0.8881,0.8556,0.8716,0.0622,0.8749,0.0,82.1082,3.5879
CatBoost,0.9807,0.9005,0.8598,0.8797,0.0496,0.8855,0.0,252.1313,0.1683


In [ ]:
print("trips in train:", tr.trip_id.nunique(), "| rows:", len(tr))   # expect 80 and ~2 million

trips in train: 80 | rows: 1966581


In [ ]:
# Feature importance
imp = pd.DataFrame({"feature": FEATURES, "LightGBM": m_lgb.feature_importances_ / m_lgb.feature_importances_.sum(),
                    "CatBoost": m_cb.get_feature_importance() / 100}).sort_values("CatBoost", ascending=False)
imp.round(3)

,feature,LightGBM,CatBoost
10,speed_mps,0.093,0.169
5,proj_frac,0.084,0.149
6,arc_len_m,0.196,0.099
8,head_diff_deg,0.045,0.085
2,dist_rel_m,0.062,0.083
4,n_cands,0.066,0.056
16,deg_from,0.051,0.053
9,head_cos,0.045,0.051
17,deg_to,0.054,0.046
14,dist_next_pt_m,0.051,0.042


In [ ]:
KS = (-10, -5, -3, -2, -1, 1, 2, 3, 5, 10)
def add_ctx(df):
    d = df.copy()
    idx = d.set_index(["trip_id", "observation_index", "arc_id"]).dist_m; idx = idx[~idx.index.duplicated()]
    best = d[d.rank_dist == 0].set_index(["trip_id", "observation_index"]).arc_id
    new = {}
    for k in KS:
        key = pd.MultiIndex.from_arrays([d.trip_id, d.observation_index + k, d.arc_id])
        new[f"d_{k}"] = idx.reindex(key).fillna(100).values            # distance of the same arc to point i+k
        nb = best.reindex(pd.MultiIndex.from_arrays([d.trip_id, d.observation_index + k])).values
        new[f"nearest_{k}"] = (nb == d.arc_id.values).astype(np.int8)  # is this arc the nearest at i+k?
    N = pd.DataFrame(new, index=d.index); dc = [f"d_{k}" for k in KS]; nc = [f"nearest_{k}" for k in KS]
    N["d_nbmean"] = N[dc].mean(axis=1); N["d_nbmax"] = N[dc].max(axis=1)
    N["d_near_mean"] = N[[f"d_{k}" for k in (-2, -1, 1, 2)]].mean(axis=1)
    N["n_nearest"] = N[nc].sum(axis=1); N["n_within30"] = (N[dc] < 30).sum(axis=1)
    return pd.concat([d, N], axis=1)

tr2, te2 = add_ctx(tr), add_ctx(te)
F2 = FEATURES + [c for c in tr2.columns if c.startswith(("d_", "nearest_", "n_nearest", "n_within"))]
m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
                       colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1).fit(tr2[F2], tr2.label)
p = m.predict_proba(te2[F2])[:, 1]
print("Top-1 / candidate miss:", top1_and_miss(te2, p))

Top-1 / candidate miss: (np.float64(0.8931688705378142), 0.0)


In [ ]:
os.makedirs("outputs", exist_ok=True); res.to_csv("outputs/model_results.csv")
pd.DataFrame({"trip_id": te.trip_id, "observation_index": te.observation_index, "arc_id": te.arc_id,
              "p_lgb": p_lgb, "p_cb": p_cb}).to_csv("outputs/student_test_scores.csv.gz", index=False, compression="gzip")
print("saved outputs/")